# Pruned Dijkstra – Test Notebook

Loads **test1** locations, queries the road network from the database,
runs the Steiner solver, and shows four layers on an interactive map:

| Layer | Colour |
|---|---|
| Road network (full queried area) | grey |
| Input locations | blue |
| Solution nodes (terminals + Steiner points) | red / orange |
| Solution edges | green |

In [ ]:
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
import geopandas as gpd
import fiona

from steiner_aio.components.read_data import (
    check_geometry_type,
    put_crs,
    query_for_road_network,
    output_hadndler,
)
from steiner_aio.components.preprocessing import (
    create_polygon,
    create_linering_from_polygon,
)
from src.pipeline.runner import SteinerRunner

print('Imports OK')

## 1 – Load locations

In [ ]:
project_root = Path('.').resolve()
location_src = project_root / 'data_examples' / 'Albany OLT-01 UG Locations.sqlite' #'test3' / 'test3 locations.sqlite'

# Auto-detect layer name (SpatiaLite)
layer_name = fiona.listlayers(str(location_src))[0]
locations_raw = gpd.read_file(str(location_src), layer=layer_name)

print(f'Loaded {len(locations_raw)} locations')
print(f'CRS: {locations_raw.crs}')
locations_raw.head(3)

In [ ]:
# Validate geometry and set/fix CRS
results_dir = project_root / 'outputs' / 'test_notebook'
results_dir.mkdir(parents=True, exist_ok=True)

locations = check_geometry_type(locations_raw, str(results_dir))
locations = put_crs(locations, str(results_dir))  # → ESRI:102008

print(f'CRS after put_crs: {locations.crs}')
print(f'Centroid (projected): x={locations.geometry.x.mean():.0f}  y={locations.geometry.y.mean():.0f}')

## 2 – Query road network

In [ ]:
# Build the query polygon (convex hull + 1 km buffer so roads *between*
# far-apart locations are included – see fix applied to create_polygon)
polygon = create_polygon(locations, buffer_distance=1000)
line_string_kml = create_linering_from_polygon(polygon)

print('Querying road network …')
output = query_for_road_network(
    input=line_string_kml,
    database='wiroidb2',
    user='postgresqlwireless2020',
    password='software2020!!',
    host='wirelesspostgresqlflexible.postgres.database.azure.com',
    port='5432',
    function_name='ww_get_all_roads_in_poly',
    path_for_log=str(results_dir),
)

roads = output_hadndler(output)   # → ESRI:102008
print(f'Retrieved {len(roads)} road segments')
print(f'Roads CRS: {roads.crs}')

## 3 – Reproject to WGS-84 and save queried roads

In [ ]:
locations_wgs = locations.to_crs('EPSG:4326')
roads_wgs     = roads.to_crs('EPSG:4326')

# Persist for reference
roads_wgs.to_file(results_dir / 'queried_roads.gpkg', layer='roads', driver='GPKG')

print(f'Locations: {len(locations_wgs)}')
print(f'Roads:     {len(roads_wgs)}')

## 4 – Build graph and map terminals

In [ ]:
runner = SteinerRunner()
runner.load_from_geodataframes(roads_wgs, locations_wgs)
runner.build_graph()

G = runner.graph
print(f'Graph: {G.number_of_nodes():,} nodes, {G.number_of_edges():,} edges')

In [ ]:
runner.map_terminals(max_distance_meters=100)

print(f'Mapped terminals: {len(runner.terminals)}')

## 5 – Run Pruned Dijkstra

In [ ]:
result = runner.run(algorithm='pruned_dijkstra')

print(f'\n--- Result ---')
print(f'  Terminals:      {len(runner.terminals)}')
print(f'  Steiner nodes:  {len(result.steiner_points)}')
print(f'  Solution nodes: {result.node_count}')
print(f'  Solution edges: {result.edge_count}')
print(f'  Total weight:   {result.total_weight:,.1f} m  ({result.total_weight/1609.34:.2f} mi)')
print(f'  Connected:      {result.is_connected}')
print(f'  Time:           {result.execution_time:.4f} s')

## 6 – Build GeoDataFrames for each layer

In [ ]:
from shapely.geometry import LineString, Point

# --- Solution edges ---
sol_edge_rows = []
for u, v, data in result.steiner_graph.edges(data=True):
    if 'geometry' in data:
        geom = data['geometry']
    else:
        ud, vd = result.steiner_graph.nodes[u], result.steiner_graph.nodes[v]
        geom = LineString([(ud['x'], ud['y']), (vd['x'], vd['y'])])
    sol_edge_rows.append({
        'from_node': u,
        'to_node': v,
        'weight_m': data.get('weight', 0.0),
        'geometry': geom,
    })

sol_edges_gdf = gpd.GeoDataFrame(sol_edge_rows, crs='EPSG:4326')

# --- Solution nodes (terminals + Steiner points) ---
sol_node_rows = []
for n, data in result.steiner_graph.nodes(data=True):
    if 'x' not in data:
        continue
    sol_node_rows.append({
        'node_id': n,
        'type': 'terminal' if n in result.terminals else 'steiner_point',
        'geometry': Point(data['x'], data['y']),
    })

sol_nodes_gdf = gpd.GeoDataFrame(sol_node_rows, crs='EPSG:4326')

terminals_gdf     = sol_nodes_gdf[sol_nodes_gdf['type'] == 'terminal']
steiner_pts_gdf   = sol_nodes_gdf[sol_nodes_gdf['type'] == 'steiner_point']
/Users/narekmeloyan/PycharmProjects/Steiner-AI/test_pruned_dijkstra.ipynb
print(f'Solution edges GDF:   {len(sol_edges_gdf)}')
print(f'Terminal nodes:       {len(terminals_gdf)}')
print(f'Steiner points:       {len(steiner_pts_gdf)}')

## 7 – Interactive map (Folium)

Four toggleable layers:
- **Grey** – full queried road network  
- **Blue** – input location points  
- **Green** – solution edges  
- **Red** – terminal nodes snapped to road  
- **Orange** – Steiner (junction) points

In [ ]:
import folium
from folium.plugins import MeasureControl

# Centre map on location centroid
cx = locations_wgs.geometry.x.mean()
cy = locations_wgs.geometry.y.mean()

m = folium.Map(location=[cy, cx], zoom_start=14, tiles='CartoDB positron')

def iter_lines(geom):
    """Yield LineStrings from any geometry; silently skip Points etc."""
    if geom is None or geom.is_empty:
        return
    if geom.geom_type == 'LineString':
        yield geom
    elif geom.geom_type == 'MultiLineString':
        yield from geom.geoms

# ── Road network ─────────────────────────────────────────────────────────────
road_layer = folium.FeatureGroup(name='Road network', show=True)
for _, row in roads_wgs.iterrows():
    for line in iter_lines(row.geometry):
        coords = [(lat, lon) for lon, lat in line.coords]
        folium.PolyLine(
            coords, color='#888888', weight=1.5, opacity=0.6,
            tooltip=str(row.get('road_type', ''))
        ).add_to(road_layer)
road_layer.add_to(m)

# ── Solution edges ────────────────────────────────────────────────────────────
sol_edge_layer = folium.FeatureGroup(name='Solution edges', show=True)
for _, row in sol_edges_gdf.iterrows():
    for line in iter_lines(row.geometry):
        coords = [(lat, lon) for lon, lat in line.coords]
        folium.PolyLine(
            coords, color='#00aa44', weight=4, opacity=0.9,
            tooltip=f"{row['from_node']} \u2192 {row['to_node']}  ({row['weight_m']:.1f} m)"
        ).add_to(sol_edge_layer)
sol_edge_layer.add_to(m)

# ── Input locations ───────────────────────────────────────────────────────────
loc_layer = folium.FeatureGroup(name='Input locations', show=True)
for _, row in locations_wgs.iterrows():
    geom = row.geometry
    if geom is None or geom.geom_type != 'Point':
        continue
    addr = row.get('complete_st_address', '')
    folium.CircleMarker(
        location=[geom.y, geom.x],
        radius=4, color='#2255cc', fill=True, fill_color='#4477ff',
        fill_opacity=0.8, tooltip=str(addr)
    ).add_to(loc_layer)
loc_layer.add_to(m)

# ── Terminal nodes ────────────────────────────────────────────────────────────
term_layer = folium.FeatureGroup(name='Terminal nodes (snapped)', show=True)
for _, row in terminals_gdf.iterrows():
    pt = row.geometry
    folium.CircleMarker(
        location=[pt.y, pt.x],
        radius=6, color='#cc0000', fill=True, fill_color='#ff3333',
        fill_opacity=0.9, tooltip=f'Terminal {row["node_id"]}'
    ).add_to(term_layer)
term_layer.add_to(m)

# ── Steiner (junction) points ─────────────────────────────────────────────────
steiner_layer = folium.FeatureGroup(name='Steiner junction points', show=True)
for _, row in steiner_pts_gdf.iterrows():
    pt = row.geometry
    folium.CircleMarker(
        location=[pt.y, pt.x],
        radius=5, color='#cc7700', fill=True, fill_color='#ffaa00',
        fill_opacity=0.85, tooltip=f'Steiner pt {row["node_id"]}'
    ).add_to(steiner_layer)
steiner_layer.add_to(m)

folium.LayerControl(collapsed=False).add_to(m)
MeasureControl(primary_length_unit='miles').add_to(m)

# Save and display
map_path = results_dir / 'pruned_dijkstra_result.html'
m.save(str(map_path))
print(f'Map saved \u2192 {map_path}')
m

## 8 – Static overview (Matplotlib)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

fig, ax = plt.subplots(figsize=(12, 10))

roads_wgs.plot(
    ax=ax, color='#cccccc', linewidth=0.8, label='Roads'
)
sol_edges_gdf.plot(
    ax=ax, color='#00aa44', linewidth=2.5, label='Solution edges', zorder=3
)
locations_wgs.plot(
    ax=ax, color='#4477ff', markersize=20, label='Locations', zorder=4
)
if len(steiner_pts_gdf):
    steiner_pts_gdf.plot(
        ax=ax, color='#ffaa00', markersize=25,
        marker='^', label='Steiner points', zorder=5
    )
terminals_gdf.plot(
    ax=ax, color='#ff3333', markersize=30,
    marker='o', label='Terminals (snapped)', zorder=6
)

ax.set_title(
    f'Pruned Dijkstra – Steiner tree\n'
    f'{len(runner.terminals)} terminals  |  '
    f'{result.edge_count} solution edges  |  '
    f'{result.total_weight/1609.34:.2f} mi total',
    fontsize=13
)
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
ax.legend(loc='upper right', fontsize=9)
ax.set_aspect('equal')
plt.tight_layout()

fig_path = results_dir / 'pruned_dijkstra_result.png'
plt.savefig(str(fig_path), dpi=150)
print(f'Figure saved → {fig_path}')
plt.show()

## 9 – Export solution layers

In [ ]:
paths = runner.export_solution(result, results_dir, export_format='gpkg')
for name, path in paths.items():
    print(f'  {name}: {path}')

## 10 – Metadata summary

In [ ]:
import pandas as pd

meta = result.metadata
rows = []
for comp_id, stats in meta.get('per_component', {}).items():
    rows.append({'component': comp_id, **stats})

if rows:
    df = pd.DataFrame(rows)
    print('Per-component stats:')
    display(df)

print(f"\nIsolated terminals : {meta.get('isolated_terminals', [])}")
print(f"Total boundary pairs : {meta.get('total_boundary_pairs', '—')}")
print(f"Total selected edges : {meta.get('total_selected_edges', '—')}")
print(f"Total hubs used      : {meta.get('total_hubs', '—')}")

---
## Part 2 – NY State Parquet Graph

Loads the pre-built NY state road graph from parquet files, extracts a local subgraph around the **test1 Albany locations**, snaps each location to its nearest road node, and runs Pruned Dijkstra.

In [ ]:
# ── 1. Load NY parquet files ──────────────────────────────────────────────────
import warnings; warnings.filterwarnings('ignore')
import pandas as pd
import numpy as np
import geopandas as gpd
import fiona
import networkx as nx
from pathlib import Path
from pyproj import Transformer
from scipy.spatial import cKDTree
from shapely.geometry import Point, LineString

EDGES_PATH = Path('/Users/arka/Downloads/edges_NY.parquet')
NODES_PATH = Path('/Users/arka/Downloads/nodes_NY.parquet')
PARQUET_CRS = 'ESRI:102008'

nodes_df = pd.read_parquet(NODES_PATH)
edges_df = pd.read_parquet(EDGES_PATH)
print(f"Loaded {len(nodes_df):,} nodes  |  {len(edges_df):,} edges (full NY state)")


In [ ]:
# ── 2. Extract local subgraph around test1 locations ─────────────────────────
# Re-use locations already loaded above (reprojected to ESRI:102008)
locs_proj102 = locations_wgs.to_crs(PARQUET_CRS)

xmin, ymin, xmax, ymax = locs_proj102.total_bounds
BUFFER_M = 5_000   # 5 km  (~3 miles) around the location cluster

nx_min = xmin - BUFFER_M; nx_max = xmax + BUFFER_M
ny_min = ymin - BUFFER_M; ny_max = ymax + BUFFER_M

mask_nodes = (
    (nodes_df['x'] >= nx_min) & (nodes_df['x'] <= nx_max) &
    (nodes_df['y'] >= ny_min) & (nodes_df['y'] <= ny_max)
)
local_nodes = nodes_df[mask_nodes].copy()
local_ids   = set(local_nodes['node_id'])

mask_edges = (
    edges_df['source'].isin(local_ids) & edges_df['target'].isin(local_ids)
)
local_edges = edges_df[mask_edges].copy()

print(f"Local subgraph : {len(local_nodes):,} nodes  |  {len(local_edges):,} edges")


In [ ]:
# ── 3. Build NetworkX graph ───────────────────────────────────────────────────
G_ny = nx.Graph()

for _, row in local_nodes.iterrows():
    G_ny.add_node(int(row['node_id']), x=row['x'], y=row['y'])

for _, row in local_edges.iterrows():
    G_ny.add_edge(int(row['source']), int(row['target']), weight=row['weight'])

comps = list(nx.connected_components(G_ny))
print(f"Graph : {G_ny.number_of_nodes():,} nodes  |  {G_ny.number_of_edges():,} edges")
print(f"Connected components : {len(comps)}")


In [ ]:
# ── 4. Snap locations to nearest graph node (KD-tree) ────────────────────────
# Build KD-tree over local node coordinates (projected)
node_ids  = local_nodes['node_id'].values
node_xy   = local_nodes[['x', 'y']].values
kdtree    = cKDTree(node_xy)

# Project locations to ESRI:102008 for distance search
locs_proj102 = locations_wgs.to_crs(PARQUET_CRS)
loc_xy       = np.column_stack([locs_proj102.geometry.x, locs_proj102.geometry.y])

MAX_SNAP_M = 200   # reject if nearest node is farther than this

dists, idx = kdtree.query(loc_xy, k=1)
terminals_ny: set[int] = set()
unmapped = 0
for i, (d, ni) in enumerate(zip(dists, idx)):
    if d <= MAX_SNAP_M:
        terminals_ny.add(int(node_ids[ni]))
    else:
        unmapped += 1

print(f"Locations     : {len(locs_proj102)}")
print(f"Mapped terminals : {len(terminals_ny)}")
print(f"Unmapped (>{MAX_SNAP_M}m away) : {unmapped}")


In [ ]:
# ── 5. Run Pruned Dijkstra ────────────────────────────────────────────────────
from src.algorithms.pruned_dijkstra import PrunedDijkstraSteiner

solver = PrunedDijkstraSteiner()
result_ny = solver.solve(G_ny, terminals_ny)

print(f"Terminals      : {len(terminals_ny)}")
print(f"Solution nodes : {result_ny.node_count}")
print(f"Solution edges : {result_ny.edge_count}")
print(f"Total weight   : {result_ny.total_weight:,.1f} m  ({result_ny.total_weight/1609.34:.2f} mi)")
print(f"Connected      : {result_ny.is_connected}")
print(f"Time           : {result_ny.execution_time:.4f} s")


In [ ]:
# ── 6. Build GeoDataFrames (reproject to WGS-84 for display) ─────────────────
to_wgs = Transformer.from_crs(PARQUET_CRS, 'EPSG:4326', always_xy=True)

def proj_to_wgs(x, y):
    lon, lat = to_wgs.transform(x, y)
    return lon, lat

# Solution nodes
sol_node_rows_ny = []
for n, data in result_ny.steiner_graph.nodes(data=True):
    if 'x' not in data: continue
    lon, lat = proj_to_wgs(data['x'], data['y'])
    sol_node_rows_ny.append({
        'node_id': n,
        'type': 'terminal' if n in terminals_ny else 'steiner_point',
        'geometry': Point(lon, lat),
    })
sol_nodes_ny_gdf = gpd.GeoDataFrame(sol_node_rows_ny, crs='EPSG:4326')
terminals_ny_gdf  = sol_nodes_ny_gdf[sol_nodes_ny_gdf['type'] == 'terminal']
steiner_ny_gdf    = sol_nodes_ny_gdf[sol_nodes_ny_gdf['type'] == 'steiner_point']

# Solution edges (straight lines between node pairs — no road geometry in parquet)
sol_edge_rows_ny = []
for u, v, edata in result_ny.steiner_graph.edges(data=True):
    ud = result_ny.steiner_graph.nodes[u]
    vd = result_ny.steiner_graph.nodes[v]
    if 'x' not in ud or 'x' not in vd: continue
    u_lon, u_lat = proj_to_wgs(ud['x'], ud['y'])
    v_lon, v_lat = proj_to_wgs(vd['x'], vd['y'])
    sol_edge_rows_ny.append({
        'from_node': u,
        'to_node': v,
        'weight_m': edata.get('weight', 0.0),
        'geometry': LineString([(u_lon, u_lat), (v_lon, v_lat)]),
    })
sol_edges_ny_gdf = gpd.GeoDataFrame(sol_edge_rows_ny, crs='EPSG:4326')

print(f"Solution edges GDF : {len(sol_edges_ny_gdf)}")
print(f"Terminal nodes     : {len(terminals_ny_gdf)}")
print(f"Steiner points     : {len(steiner_ny_gdf)}")


In [ ]:
# ── 7. Interactive Folium map (NY parquet result) ─────────────────────────────
import folium
from folium.plugins import MeasureControl

cx = locations_wgs.geometry.x.mean()
cy = locations_wgs.geometry.y.mean()

m_ny = folium.Map(location=[cy, cx], zoom_start=14, tiles='CartoDB positron')

def iter_lines(geom):
    if geom is None or geom.is_empty: return
    if geom.geom_type == 'LineString': yield geom
    elif geom.geom_type == 'MultiLineString': yield from geom.geoms

# Solution edges
sol_edge_layer_ny = folium.FeatureGroup(name='Solution edges (NY parquet)', show=True)
for _, row in sol_edges_ny_gdf.iterrows():
    for line in iter_lines(row.geometry):
        coords = [(lat, lon) for lon, lat in line.coords]
        folium.PolyLine(
            coords, color='#00aa44', weight=4, opacity=0.9,
            tooltip=f"{row['from_node']} → {row['to_node']}  ({row['weight_m']:.1f} m)"
        ).add_to(sol_edge_layer_ny)
sol_edge_layer_ny.add_to(m_ny)

# Input locations
loc_layer_ny = folium.FeatureGroup(name='Input locations', show=True)
for _, row in locations_wgs.iterrows():
    geom = row.geometry
    if geom is None or geom.geom_type != 'Point': continue
    folium.CircleMarker(
        location=[geom.y, geom.x], radius=4,
        color='#2255cc', fill=True, fill_color='#4477ff', fill_opacity=0.8,
        tooltip=str(row.get('complete_st_address', ''))
    ).add_to(loc_layer_ny)
loc_layer_ny.add_to(m_ny)

# Terminal nodes
term_layer_ny = folium.FeatureGroup(name='Terminal nodes', show=True)
for _, row in terminals_ny_gdf.iterrows():
    pt = row.geometry
    folium.CircleMarker(
        location=[pt.y, pt.x], radius=6,
        color='#cc0000', fill=True, fill_color='#ff3333', fill_opacity=0.9,
        tooltip=f'Terminal {row["node_id"]}'
    ).add_to(term_layer_ny)
term_layer_ny.add_to(m_ny)

# Steiner points
steiner_layer_ny = folium.FeatureGroup(name='Steiner junction points', show=True)
for _, row in steiner_ny_gdf.iterrows():
    pt = row.geometry
    folium.CircleMarker(
        location=[pt.y, pt.x], radius=5,
        color='#cc7700', fill=True, fill_color='#ffaa00', fill_opacity=0.85,
        tooltip=f'Steiner pt {row["node_id"]}'
    ).add_to(steiner_layer_ny)
steiner_layer_ny.add_to(m_ny)

folium.LayerControl(collapsed=False).add_to(m_ny)
MeasureControl(primary_length_unit='miles').add_to(m_ny)

results_dir_ny = Path('outputs/test_notebook')
map_path_ny = results_dir_ny / 'pruned_dijkstra_ny_parquet.html'
m_ny.save(str(map_path_ny))
print(f'Map saved → {map_path_ny}')
m_ny